# Advanced Queries

Uses cleaned data from **../../Part 2/cleaned_data** and the SQLite database in **../database/ecommerce.db**.

In [1]:


from pathlib import Path
import sqlite3
import pandas as pd

DB_PATH = Path("../database/ecommerce.db")

def run_query(conn, sql):
    return pd.read_sql_query(sql, conn)

REVENUE_EXPR = "oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)"


def query_7_running_total_revenue_per_region(conn):
    sql = f"""
        WITH daily AS (
            SELECT
                o.region_code,
                date(o.order_date) AS order_day,
                SUM({REVENUE_EXPR}) AS daily_revenue
            FROM order_items oi
            JOIN orders o ON oi.order_id = o.order_id
            GROUP BY o.region_code, order_day
        )
        SELECT
            region_code,
            order_day AS order_date,
            ROUND(daily_revenue, 2) AS daily_revenue,
            ROUND(
                SUM(daily_revenue) OVER (
                    PARTITION BY region_code ORDER BY order_day
                    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                ), 2
            ) AS running_total
        FROM daily
        ORDER BY region_code, order_day;
    """
    return run_query(conn, sql)


def query_8_dense_rank_products_by_category(conn):
    sql = f"""
        WITH product_revenue AS (
            SELECT
                p.category,
                p.product_id,
                p.product_name,
                SUM({REVENUE_EXPR}) AS total_revenue
            FROM order_items oi
            JOIN products p ON oi.product_id = p.product_id
            JOIN orders o   ON oi.order_id   = o.order_id
            GROUP BY p.category, p.product_id, p.product_name
        )
        SELECT
            category,
            product_name,
            ROUND(total_revenue, 2) AS total_revenue,
            DENSE_RANK() OVER (PARTITION BY category ORDER BY total_revenue DESC) AS rank_in_category
        FROM product_revenue
        ORDER BY category, rank_in_category;
    """
    return run_query(conn, sql)


def query_9_lag_days_between_orders(conn):
    """
    'At Risk' is computed as a window aggregate (AVG days_gap per customer)
    rather than a separate rollup query, so every row already carries its
    customer's overall risk status alongside the individual gap.
    """
    sql = """
        WITH ordered AS (
            SELECT
                customer_id,
                order_date,
                LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS previous_order_date
            FROM orders
            WHERE customer_id IS NOT NULL
        ),
        gaps AS (
            SELECT
                customer_id,
                order_date,
                previous_order_date,
                CASE WHEN previous_order_date IS NULL THEN NULL
                     ELSE julianday(order_date) - julianday(previous_order_date)
                END AS days_gap
            FROM ordered
        )
        SELECT
            CAST(customer_id AS INTEGER) AS customer_id,
            order_date,
            previous_order_date,
            ROUND(days_gap, 2) AS days_gap,
            CASE WHEN AVG(days_gap) OVER (PARTITION BY customer_id) > 30
                 THEN 'At Risk' ELSE 'Regular' END AS customer_status
        FROM gaps
        ORDER BY customer_id, order_date;
    """
    return run_query(conn, sql)


def query_10_multilevel_cte_revenue_tiers(conn):
    sql = f"""
        WITH monthly_customer_revenue AS (
            SELECT
                o.customer_id,
                strftime('%Y-%m', o.order_date) AS year_month,
                SUM({REVENUE_EXPR}) AS monthly_revenue
            FROM order_items oi
            JOIN orders o ON oi.order_id = o.order_id
            WHERE o.customer_id IS NOT NULL
            GROUP BY o.customer_id, year_month
        ),
        categorized AS (
            SELECT
                year_month,
                customer_id,
                CASE
                    WHEN monthly_revenue > 10000 THEN 'High'
                    WHEN monthly_revenue >= 5000  THEN 'Medium'
                    ELSE 'Low'
                END AS revenue_tier
            FROM monthly_customer_revenue
        )
        SELECT year_month, revenue_tier, COUNT(*) AS customer_count
        FROM categorized
        GROUP BY year_month, revenue_tier
        ORDER BY year_month, revenue_tier;
    """
    return run_query(conn, sql)


def query_11_ntile_quartile_segmentation(conn):
    sql = f"""
        WITH customer_ltv AS (
            SELECT
                o.customer_id,
                SUM({REVENUE_EXPR}) AS total_value
            FROM order_items oi
            JOIN orders o ON oi.order_id = o.order_id
            WHERE o.customer_id IS NOT NULL
            GROUP BY o.customer_id
        ),
        ranked AS (
            SELECT
                customer_id,
                total_value,
                NTILE(4) OVER (ORDER BY total_value DESC) AS quartile
            FROM customer_ltv
        )
        SELECT
            CAST(customer_id AS INTEGER) AS customer_id,
            ROUND(total_value, 2) AS total_value,
            quartile,
            CASE quartile
                WHEN 1 THEN 'Platinum'
                WHEN 2 THEN 'Gold'
                WHEN 3 THEN 'Silver'
                ELSE 'Bronze'
            END AS quartile_label
        FROM ranked
        ORDER BY total_value DESC;
    """
    return run_query(conn, sql)


def query_12_year_over_year_comparison(conn):
    sql = f"""
        WITH monthly_rev AS (
            SELECT
                CAST(strftime('%Y', o.order_date) AS INTEGER) AS year,
                CAST(strftime('%m', o.order_date) AS INTEGER) AS month,
                SUM({REVENUE_EXPR}) AS revenue
            FROM order_items oi
            JOIN orders o ON oi.order_id = o.order_id
            GROUP BY year, month
        )
        SELECT
            year,
            month,
            ROUND(revenue, 2) AS revenue,
            ROUND(LAG(revenue) OVER (PARTITION BY month ORDER BY year), 2) AS prev_year_revenue,
            ROUND(
                100.0 * (revenue - LAG(revenue) OVER (PARTITION BY month ORDER BY year))
                / NULLIF(LAG(revenue) OVER (PARTITION BY month ORDER BY year), 0),
                2
            ) AS yoy_growth_percent
        FROM monthly_rev
        ORDER BY year, month;
    """
    return run_query(conn, sql)


def query_13_first_last_category_shift(conn):
    sql = """
        WITH customer_category_events AS (
            SELECT
                o.customer_id,
                o.order_date,
                p.category,
                FIRST_VALUE(p.category) OVER (
                    PARTITION BY o.customer_id ORDER BY o.order_date
                    ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
                ) AS first_category,
                LAST_VALUE(p.category) OVER (
                    PARTITION BY o.customer_id ORDER BY o.order_date
                    ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
                ) AS last_category
            FROM order_items oi
            JOIN orders o    ON oi.order_id   = o.order_id
            JOIN products p  ON oi.product_id = p.product_id
            WHERE o.customer_id IS NOT NULL
        )
        SELECT DISTINCT
            CAST(customer_id AS INTEGER) AS customer_id,
            first_category,
            last_category,
            CASE WHEN first_category != last_category THEN 'Yes' ELSE 'No' END AS category_shift
        FROM customer_category_events
        ORDER BY customer_id;
    """
    return run_query(conn, sql)


def query_14_cumulative_revenue_distribution(conn):
    sql = f"""
        WITH customer_ltv AS (
            SELECT
                o.customer_id,
                SUM({REVENUE_EXPR}) AS revenue
            FROM order_items oi
            JOIN orders o ON oi.order_id = o.order_id
            WHERE o.customer_id IS NOT NULL
            GROUP BY o.customer_id
        ),
        ranked AS (
            SELECT
                customer_id,
                revenue,
                SUM(revenue) OVER (
                    ORDER BY revenue DESC
                    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                ) AS cumulative_revenue,
                SUM(revenue) OVER () AS grand_total_revenue
            FROM customer_ltv
        )
        SELECT
            CAST(customer_id AS INTEGER) AS customer_id,
            ROUND(revenue, 2) AS revenue,
            ROUND(cumulative_revenue, 2) AS cumulative_revenue,
            ROUND(100.0 * cumulative_revenue / grand_total_revenue, 2) AS cumulative_percent
        FROM ranked
        ORDER BY revenue DESC;
    """
    return run_query(conn, sql)


def query_15_cohort_retention(conn):
    """
    Cohort = the customer's registration month. month_offset is computed by
    converting each YYYY-MM into an absolute month index (year*12 + month)
    and subtracting, rather than hardcoding day/date arithmetic, so this
    keeps working regardless of how many months the dataset spans.
    """
    sql = """
        WITH cohorts AS (
            SELECT customer_id, strftime('%Y-%m', registration_date) AS cohort_month
            FROM customers
        ),
        customer_order_months AS (
            SELECT DISTINCT customer_id, strftime('%Y-%m', order_date) AS order_month
            FROM orders
            WHERE customer_id IS NOT NULL
        ),
        cohort_activity AS (
            SELECT
                c.customer_id,
                c.cohort_month,
                (CAST(strftime('%Y', com.order_month || '-01') AS INTEGER) * 12
                    + CAST(strftime('%m', com.order_month || '-01') AS INTEGER))
                -
                (CAST(strftime('%Y', c.cohort_month || '-01') AS INTEGER) * 12
                    + CAST(strftime('%m', c.cohort_month || '-01') AS INTEGER)) AS month_offset
            FROM cohorts c
            JOIN customer_order_months com ON c.customer_id = com.customer_id
        ),
        cohort_sizes AS (
            SELECT cohort_month, COUNT(*) AS cohort_size
            FROM cohorts
            GROUP BY cohort_month
        )
        SELECT
            ca.cohort_month,
            cs.cohort_size,
            ca.month_offset,
            COUNT(DISTINCT ca.customer_id) AS active_customers,
            ROUND(100.0 * COUNT(DISTINCT ca.customer_id) / cs.cohort_size, 2) AS retention_rate_percent
        FROM cohort_activity ca
        JOIN cohort_sizes cs ON ca.cohort_month = cs.cohort_month
        WHERE ca.month_offset BETWEEN 0 AND 3
        GROUP BY ca.cohort_month, ca.month_offset
        ORDER BY ca.cohort_month, ca.month_offset;
    """
    return run_query(conn, sql)


def query_16_frequently_bought_together(conn):
    """
    Self-join on order_id with product_id_a < product_id_b, which
    simultaneously excludes a product pairing with itself and de-duplicates
    (A,B)/(B,A) into a single row.
    """
    sql = """
        WITH order_products AS (
            SELECT DISTINCT order_id, product_id
            FROM order_items
            WHERE quantity > 0
        ),
        pairs AS (
            SELECT
                a.product_id AS product_id_a,
                b.product_id AS product_id_b,
                COUNT(*) AS times_bought_together
            FROM order_products a
            JOIN order_products b
                ON a.order_id = b.order_id
                AND a.product_id < b.product_id
            GROUP BY a.product_id, b.product_id
        )
        SELECT
            pa.product_name AS product_a,
            pb.product_name AS product_b,
            pairs.times_bought_together
        FROM pairs
        JOIN products pa ON pairs.product_id_a = pa.product_id
        JOIN products pb ON pairs.product_id_b = pb.product_id
        ORDER BY pairs.times_bought_together DESC
        LIMIT 20;
    """
    return run_query(conn, sql)


def main():
    conn = sqlite3.connect(DB_PATH)

    print("\n=== Query 7: Running Total Revenue per Region (sample) ===")
    r7 = query_7_running_total_revenue_per_region(conn)
    print(f"({len(r7)} region-day rows total)")
    print(r7[r7.region_code == r7.region_code.iloc[0]].head(5).to_string(index=False))
    print("...")
    print(r7[r7.region_code == r7.region_code.iloc[0]].tail(3).to_string(index=False))

    print("\n=== Query 8: DENSE_RANK Products by Category (top 3 per category) ===")
    r8 = query_8_dense_rank_products_by_category(conn)
    print(r8[r8.rank_in_category <= 3].to_string(index=False))

    print("\n=== Query 9: LAG Days Between Orders (sample) ===")
    r9 = query_9_lag_days_between_orders(conn)
    print(f"({len(r9)} order rows)")
    at_risk_customers = r9[r9.customer_status == "At Risk"].customer_id.nunique()
    print(f"Customers flagged 'At Risk': {at_risk_customers}")
    print(r9.head(8).to_string(index=False))

    print("\n=== Query 10: Multi-level CTE - Monthly Revenue Tiers (sample) ===")
    r10 = query_10_multilevel_cte_revenue_tiers(conn)
    print(r10.head(12).to_string(index=False))

    print("\n=== Query 11: NTILE Quartile Segmentation (sample) ===")
    r11 = query_11_ntile_quartile_segmentation(conn)
    print(r11.head(5).to_string(index=False))
    print("...")
    print(r11.tail(5).to_string(index=False))

    print("\n=== Query 12: Year-over-Year Comparison ===")
    r12 = query_12_year_over_year_comparison(conn)
    print(r12.to_string(index=False))

    print("\n=== Query 13: First/Last Category Shift (sample) ===")
    r13 = query_13_first_last_category_shift(conn)
    print(f"({len(r13)} customers, {(r13.category_shift == 'Yes').sum()} shifted category)")
    print(r13.head(8).to_string(index=False))

    print("\n=== Query 14: Cumulative Revenue Distribution (sample) ===")
    r14 = query_14_cumulative_revenue_distribution(conn)
    print(r14.head(10).to_string(index=False))
    top10pct_cutoff = int(len(r14) * 0.10)
    print(f"\nTop 10% of customers ({top10pct_cutoff}) contribute "
          f"{r14.iloc[top10pct_cutoff - 1]['cumulative_percent']}% of total revenue")

    print("\n=== Query 15: Cohort Retention Analysis (sample) ===")
    r15 = query_15_cohort_retention(conn)
    print(r15.head(16).to_string(index=False))

    print("\n=== Query 16: Frequently Bought Together (top 20) ===")
    r16 = query_16_frequently_bought_together(conn)
    print(r16.to_string(index=False))

    conn.close()


if __name__ == "__main__":
    main()



=== Query 7: Running Total Revenue per Region (sample) ===
(1833 region-day rows total)
region_code order_date  daily_revenue  running_total
    CENTRAL 2024-05-31        5811.11        5811.11
    CENTRAL 2024-06-01       46528.54       52339.65
    CENTRAL 2024-06-07       38277.21       90616.86
    CENTRAL 2024-06-11       50541.59      141158.45
    CENTRAL 2024-06-12        3046.33      144204.79
...
region_code order_date  daily_revenue  running_total
    CENTRAL 2026-07-29       15589.56    19855437.11
    CENTRAL 2026-08-30      202808.20    20058245.31
    CENTRAL 2026-09-09       22102.16    20080347.48

=== Query 8: DENSE_RANK Products by Category (top 3 per category) ===
   category             product_name  total_revenue  rank_in_category
     Beauty         Drift Cream Plus      130192.77                 1
     Beauty         Vibe Perfume Pro      112435.25                 2
     Beauty        Aero Lipstick Max      106000.18                 3
      Books       Drift Te